In [1]:
# # 05 - Perbandingan SVM, Random Forest, dan Regresi Logistik
# Dua tahap validasi:
# 
# **A. Pada label gold 1.000 data (manusia)** - split 80:20 stratified, GridSearchCV 5-fold, plus validasi silang 5-fold pada seluruh gold. Ini perbandingan yang paling adil karena label dari manusia.
# 
# **B. Setelah seluruh dataset berlabel (~58 ribu pasangan)** - model dilatih pada data berlabel hasil pelabelan (tanpa 1.000 gold), lalu diuji pada **1.000 gold (label manusia)** sebagai uji eksternal. Ditambah uji internal split 80:20 pada data berlabel model.
# 
# > Peringatan: label ~57 ribu data dihasilkan oleh SVM (notebook 04), sehingga uji internal tahap B cenderung menguntungkan SVM (sirkular). Gunakan tahap A dan uji eksternal B sebagai dasar perbandingan.
# 
# Empat model dibandingkan: SVM, Random Forest, Regresi Logistik, dan **Regresi Linear** one-vs-rest: satu regresi linear (OLS) per kelas pada target indikator 0/1, lalu kelas dengan skor tertinggi dipilih. Tidak ada `class_weight` dan tidak ada parameter yang dicari, karena ini sengaja dijadikan pembanding dasar.
# 
# - Input : `hasil_validasi/label_gold_1000.csv`, `hasil_validasi/sentimen_kalimat_label_baru.csv`
# - Output: `hasil_validasi/perbandingan_model.xlsx`


# ===== SEL 1 (salin ke satu sel notebook) =====
import os, re, itertools, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

FOLDER_DATA  = "validasi data"             # berkas validasi_*.xlsx dan aspek_kalimat.csv
FOLDER_HASIL = "hasil_validasi"   # semua keluaran disimpan di sini
os.makedirs(FOLDER_HASIL, exist_ok=True)

SEED = 42
LABEL = ["positif", "negatif", "netral"]
ASPEK = ["wifi", "harga", "suasana", "pelayanan"]
N_PENILAI = 3

def tafsir_kappa(k):
    """Skala Landis & Koch (1977)."""
    if k is None or np.isnan(k): return "tidak dapat dihitung"
    if k < 0:     return "buruk (di bawah kebetulan)"
    if k <= 0.20: return "sangat rendah"
    if k <= 0.40: return "rendah"
    if k <= 0.60: return "sedang"
    if k <= 0.80: return "baik (substantial)"
    return "sangat baik (almost perfect)"

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split, cross_validate
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, cohen_kappa_score)


# ## Regresi Linear sebagai pengklasifikasi


# ===== SEL 2 (salin ke satu sel notebook) =====
class RegresiLinearKlasifikasi(BaseEstimator, ClassifierMixin):
    """Regresi linear (OLS) one-vs-rest untuk klasifikasi 3 kelas:
    satu regresi per kelas pada target indikator 0/1, prediksi = kelas dengan skor tertinggi."""
    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_ = np.array(LABEL)
        T = np.column_stack([(y == c).astype(float) for c in LABEL])
        self.reg_ = LinearRegression().fit(X, T)
        return self
    def predict(self, X):
        return self.classes_[np.argmax(self.reg_.predict(X), axis=1)]


# ## Data


# ===== SEL 3 (salin ke satu sel notebook) =====
gold = pd.read_csv(f"{FOLDER_HASIL}/label_gold_1000.csv")[["id_aspek_kalimat", "label_gold"]]
ak = pd.read_csv(f"{FOLDER_HASIL}/sentimen_kalimat_label_baru.csv")
teks_model = lambda d: "aspek_" + d["aspek"] + " " + d["token_stem"].fillna("")
dg = ak.merge(gold, on="id_aspek_kalimat")
Xg, yg = teks_model(dg), dg["label_gold"]
print("gold:", len(dg), "| seluruh data:", len(ak))


# ## A. Perbandingan pada label gold


# ===== SEL 4 (salin ke satu sel notebook) =====
tf = lambda: TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
MODEL = {
 "SVM": (Pipeline([("tfidf", tf()), ("clf", SVC(class_weight="balanced", random_state=SEED))]),
         {"clf__kernel": ["linear", "poly", "rbf"], "clf__C": [0.1, 1, 10]}),
 "Random Forest": (Pipeline([("tfidf", tf()), ("clf", RandomForestClassifier(class_weight="balanced", random_state=SEED, n_jobs=-1))]),
         {"clf__n_estimators": [100, 300], "clf__max_depth": [None, 20]}),
 "Regresi Logistik": (Pipeline([("tfidf", tf()), ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED))]),
         {"clf__C": [0.1, 1, 10, 100]}),
 "Regresi Linear": (Pipeline([("tfidf", tf()), ("clf", RegresiLinearKlasifikasi())]), {}),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
strat = dg["aspek"] + "_" + yg
strat = strat.where(strat.map(strat.value_counts()) >= 10, "lain_" + yg)
Xtr, Xte, ytr, yte = train_test_split(Xg, yg, test_size=0.2, stratify=strat, random_state=SEED)
print("latih:", len(Xtr), "uji:", len(Xte))

hasilA, terbaik, pred_uji = [], {}, {}
for nama, (pipa, grid) in MODEL.items():
    gs = GridSearchCV(pipa, grid, cv=cv, scoring="f1_macro", n_jobs=-1).fit(Xtr, ytr)
    p = gs.predict(Xte); terbaik[nama] = gs; pred_uji[nama] = p
    pr, rc, f1, _ = precision_recall_fscore_support(yte, p, labels=LABEL, average="macro", zero_division=0)
    cvr = cross_validate(gs.best_estimator_, Xg, yg, cv=cv, scoring=["accuracy", "f1_macro"])
    hasilA.append({"model": nama, "param_terbaik": str(gs.best_params_), "f1_macro_cv_latih": gs.best_score_,
                   "akurasi_uji": accuracy_score(yte, p), "presisi_macro_uji": pr, "recall_macro_uji": rc, "f1_macro_uji": f1,
                   "kappa_vs_gold_uji": cohen_kappa_score(yte, p),
                   "akurasi_cv5_gold": cvr["test_accuracy"].mean(), "f1_macro_cv5_gold": cvr["test_f1_macro"].mean(),
                   "f1_macro_cv5_sd": cvr["test_f1_macro"].std()})
    print("\n==", nama, gs.best_params_)
    print(classification_report(yte, p, labels=LABEL, digits=3, zero_division=0))
hasilA = pd.DataFrame(hasilA)
hasilA.round(4)


# ===== SEL 5 (salin ke satu sel notebook) =====
kelas = []
konf = {}
for nama, p in pred_uji.items():
    pr, rc, f1, sup = precision_recall_fscore_support(yte, p, labels=LABEL, zero_division=0)
    for i, l in enumerate(LABEL):
        kelas.append({"model": nama, "kelas": l, "presisi": pr[i], "recall": rc[i], "f1": f1[i], "n": sup[i]})
    konf[nama] = pd.DataFrame(confusion_matrix(yte, p, labels=LABEL), index=["asli_" + l for l in LABEL], columns=["pred_" + l for l in LABEL])
kelas = pd.DataFrame(kelas)
kelas.pivot(index="model", columns="kelas", values="f1").round(3)


# ## B. Setelah seluruh dataset berlabel


# ===== SEL 6 (salin ke satu sel notebook) =====
# Latih pada data berlabel model (tanpa 1.000 gold); uji eksternal pada gold (label manusia)
lat = ak[ak["sumber_label"] == "model_svm"].reset_index(drop=True)
Xl, yl = teks_model(lat), lat["sentimen"]
# Data besar: SVM memakai kernel linear (kernel terbaik pada tahap A) dengan LinearSVC agar tetap cepat;
# Random Forest dibatasi fitur agar memori wajar.
tfB = lambda: TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_features=100000)
MODEL_B = {
 "SVM (linear)": Pipeline([("tfidf", tfB()), ("clf", LinearSVC(C=1, class_weight="balanced", random_state=SEED))]),
 "Random Forest": Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=3, max_features=20000)),
                            ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=SEED))]),
 "Regresi Logistik": Pipeline([("tfidf", tfB()), ("clf", LogisticRegression(C=10, class_weight="balanced", max_iter=2000, random_state=SEED))]),
 "Regresi Linear": Pipeline([("tfidf", tfB()), ("clf", RegresiLinearKlasifikasi())]),
}
Xa, Xb, ya, yb = train_test_split(Xl, yl, test_size=0.2, stratify=lat["aspek"] + "_" + yl, random_state=SEED)
hasilB, konfB = [], {}
for nama, m in MODEL_B.items():
    m.fit(Xa, ya)
    p_int = m.predict(Xb); p_eks = m.predict(Xg)
    pr_i, rc_i, f1_i, _ = precision_recall_fscore_support(yb, p_int, labels=LABEL, average="macro", zero_division=0)
    pr_e, rc_e, f1_e, _ = precision_recall_fscore_support(yg, p_eks, labels=LABEL, average="macro", zero_division=0)
    hasilB.append({"model": nama,
        "akurasi_internal": accuracy_score(yb, p_int), "f1_macro_internal": f1_i,
        "akurasi_eksternal_gold": accuracy_score(yg, p_eks), "presisi_macro_eksternal": pr_e,
        "recall_macro_eksternal": rc_e, "f1_macro_eksternal": f1_e, "kappa_eksternal_vs_gold": cohen_kappa_score(yg, p_eks)})
    konfB[nama] = pd.DataFrame(confusion_matrix(yg, p_eks, labels=LABEL), index=["gold_" + l for l in LABEL], columns=["pred_" + l for l in LABEL])
    print(nama, "selesai")
hasilB = pd.DataFrame(hasilB)
hasilB.round(4)


# ===== SEL 7 (salin ke satu sel notebook) =====
for nama, k in konfB.items():
    print("Uji eksternal (gold) -", nama); display(k); print()


# ## Kesimpulan otomatis dan penyimpanan


# ===== SEL 8 (salin ke satu sel notebook) =====
print("Tahap A (gold, F1-macro CV 5-lipat):")
print(hasilA.sort_values("f1_macro_cv5_gold", ascending=False)[["model", "akurasi_cv5_gold", "f1_macro_cv5_gold", "f1_macro_cv5_sd"]].round(4).to_string(index=False))
print("\nTahap B (uji eksternal pada gold):")
print(hasilB.sort_values("f1_macro_eksternal", ascending=False)[["model", "akurasi_eksternal_gold", "f1_macro_eksternal"]].round(4).to_string(index=False))
with pd.ExcelWriter(f"{FOLDER_HASIL}/perbandingan_model.xlsx") as xw:
    hasilA.to_excel(xw, sheet_name="A_gold_ringkasan", index=False)
    kelas.to_excel(xw, sheet_name="A_per_kelas", index=False)
    for n, k in konf.items(): k.to_excel(xw, sheet_name=("A_konfusi_" + n.replace(" ", "_").replace("(", "").replace(")", ""))[:31])
    hasilB.to_excel(xw, sheet_name="B_dataset_ringkasan", index=False)
    for n, k in konfB.items(): k.to_excel(xw, sheet_name=("B_konfusi_" + n.replace(" ", "_").replace("(", "").replace(")", ""))[:31])

gold: 987 | seluruh data: 58643
latih: 789 uji: 198

== SVM {'clf__C': 1, 'clf__kernel': 'linear'}
              precision    recall  f1-score   support

     positif      0.928     0.935     0.932       124
     negatif      0.875     0.933     0.903        60
      netral      0.556     0.357     0.435        14

    accuracy                          0.894       198
   macro avg      0.786     0.742     0.757       198
weighted avg      0.886     0.894     0.888       198


== Random Forest {'clf__max_depth': 20, 'clf__n_estimators': 300}
              precision    recall  f1-score   support

     positif      0.882     0.903     0.892       124
     negatif      0.847     0.833     0.840        60
      netral      0.250     0.214     0.231        14

    accuracy                          0.833       198
   macro avg      0.660     0.650     0.655       198
weighted avg      0.827     0.833     0.830       198


== Regresi Logistik {'clf__C': 10}
              precision    recall  f

,pred_positif,pred_negatif,pred_netral
gold_positif,597,15,6
gold_negatif,4,287,6
gold_netral,3,12,57



Uji eksternal (gold) - Random Forest


,pred_positif,pred_negatif,pred_netral
gold_positif,578,29,11
gold_negatif,14,275,8
gold_netral,6,13,53



Uji eksternal (gold) - Regresi Logistik


,pred_positif,pred_negatif,pred_netral
gold_positif,585,21,12
gold_negatif,4,285,8
gold_netral,1,11,60



Uji eksternal (gold) - Regresi Linear


,pred_positif,pred_negatif,pred_netral
gold_positif,539,64,15
gold_negatif,72,209,16
gold_netral,14,10,48



Tahap A (gold, F1-macro CV 5-lipat):
           model  akurasi_cv5_gold  f1_macro_cv5_gold  f1_macro_cv5_sd
Regresi Logistik            0.8744             0.7383           0.0216
             SVM            0.8683             0.7199           0.0183
   Random Forest            0.8186             0.7044           0.0167
  Regresi Linear            0.8318             0.6591           0.0485

Tahap B (uji eksternal pada gold):
           model  akurasi_eksternal_gold  f1_macro_eksternal
    SVM (linear)                  0.9534              0.9083
Regresi Logistik                  0.9422              0.8955
   Random Forest                  0.9179              0.8608
  Regresi Linear                  0.8065              0.7412
